In [1]:
import geopandas as gpd
import pandas as pd

In [2]:
# Load the data
unfaelle = gpd.read_file(r"C:\Users\leoni\Documents\HSLU\5. Semester\Informationsvisualisierung\Einzelprojekt\informationsvisualisierung-leonid-wouters\Daten\unfaelle_zuerich_2023_temporegime.json")

In [3]:
unfaelle.head()

,UnfallID,Unfallart,Unfallschwere,Fussgänger_beteiligt,Fahrrad_beteiligt,Motorrad_beteiligt,Strassentyp,Unfallort_Lon,Unfallort_Lat,Kanton,Gemeinde,Jahr,Monat,Wochentag,Stunde,Stunde_text,Temporegime,geometry
0,2AF2FBACB62B00DAE0530A8394273CC3,Fussgängerunfall,Unfall mit Leichtverletzten,true,true,false,Nebenstrasse,2681186,1247896,ZH,0261,2016,Januar,Mittwoch,16,16h-17h,T50,POINT (8.51365 47.37690)
1,2B51E1C5A119015CE0530A839427323A,Schleuder- oder Selbstunfall,Unfall mit Leichtverletzten,false,true,false,Hauptstrasse,2682739,1244360,ZH,0261,2016,Januar,Freitag,13,13h-14h,T50,POINT (8.53356 47.34490)
2,2C6EC4B16319006EE0530A8394275C3E,Schleuder- oder Selbstunfall,Unfall mit Leichtverletzten,false,true,false,Nebenstrasse,2683598,1248099,ZH,0261,2016,Februar,Samstag,19,19h-20h,T50,POINT (8.54563 47.37842)
3,2F5393B75558017AE0530A839427C6B4,Auffahrunfall,Unfall mit Leichtverletzten,false,false,false,Hauptstrasse,2682564,1251648,ZH,0261,2016,März,Freitag,17,17h-18h,T50,POINT (8.53259 47.41047)
4,303090D5AB8300A8E0530A83942731D6,Auffahrunfall,Unfall mit Leichtverletzten,false,false,false,Hauptstrasse,2685438,1251781,ZH,0261,2016,Januar,Donnerstag,10,10h-11h,T50,POINT (8.57069 47.41130)


In [4]:
# Load the data
beleuchtung = gpd.read_file(r"C:\Users\leoni\Documents\HSLU\5. Semester\Informationsvisualisierung\Einzelprojekt\informationsvisualisierung-leonid-wouters\Daten\beleuchtung.geojson")

In [5]:
beleuchtung.head()

,id,art,art_txt,nisnr,objid,orientierung,geometry
0,ewz_brennstelle_p.1,0,Beleuchtung,LEU118741,1,20.8,POINT (8.54326 47.43218)
1,ewz_brennstelle_p.2,0,Beleuchtung,LEU118711,2,265.7,POINT (8.53787 47.43137)
2,ewz_brennstelle_p.3,0,Beleuchtung,LEU118712,3,263.9,POINT (8.53754 47.43138)
3,ewz_brennstelle_p.4,0,Beleuchtung,LEU118713,4,58.8,POINT (8.53708 47.43143)
4,ewz_brennstelle_p.5,0,Beleuchtung,LEU118680,5,85.8,POINT (8.53817 47.43130)


In [6]:
import plotly.express as px

# Extract latitude and longitude directly from the geometry in the plot function
fig = px.scatter_mapbox(
    beleuchtung,
    lat=beleuchtung.geometry.y,  # Directly using geometry to get y-coordinates
    lon=beleuchtung.geometry.x,    # Directly using geometry to get x-coordinates
    hover_name='art_txt',
    hover_data=['art_txt'],        # Display 'art_txt' in the hover information
    zoom=10
)

fig.update_layout(mapbox_style='open-street-map')
fig.show()


In [7]:
# Umwandlung in ein passendes CRS (z.B. CH1903 / LV95 für die Schweiz)
unfaelle = unfaelle.to_crs(epsg=21781)
beleuchtung = beleuchtung.to_crs(epsg=21781)

In [8]:
import geopandas as gpd
from shapely.ops import nearest_points

# Stelle sicher, dass unfaelle und beleuchtung dasselbe Koordinatensystem verwenden
unfaelle = unfaelle.to_crs(beleuchtung.crs)

# Liste zum Speichern der Distanzen
distances = []

# Iteriere über alle Unfälle
for idx, accident in unfaelle.iterrows():
    # Finde den nächstgelegenen Punkt
    nearest = nearest_points(accident.geometry, beleuchtung.unary_union)[1]
    # Berechne die Distanz und speichere sie
    distance = accident.geometry.distance(nearest)
    distances.append(distance)

# Füge die Distanzen als neue Spalte zu unfaelle hinzu
unfaelle['distance_to_light'] = distances

# Ergebnisse überprüfen
unfaelle.head()


,UnfallID,Unfallart,Unfallschwere,Fussgänger_beteiligt,Fahrrad_beteiligt,Motorrad_beteiligt,Strassentyp,Unfallort_Lon,Unfallort_Lat,Kanton,Gemeinde,Jahr,Monat,Wochentag,Stunde,Stunde_text,Temporegime,geometry,distance_to_light
0,2AF2FBACB62B00DAE0530A8394273CC3,Fussgängerunfall,Unfall mit Leichtverletzten,true,true,false,Nebenstrasse,2681186,1247896,ZH,0261,2016,Januar,Mittwoch,16,16h-17h,T50,POINT (681186.001 247896.001),1.563390
1,2B51E1C5A119015CE0530A839427323A,Schleuder- oder Selbstunfall,Unfall mit Leichtverletzten,false,true,false,Hauptstrasse,2682739,1244360,ZH,0261,2016,Januar,Freitag,13,13h-14h,T50,POINT (682739.001 244360.001),4.428940
2,2C6EC4B16319006EE0530A8394275C3E,Schleuder- oder Selbstunfall,Unfall mit Leichtverletzten,false,true,false,Nebenstrasse,2683598,1248099,ZH,0261,2016,Februar,Samstag,19,19h-20h,T50,POINT (683598.001 248099.001),5.570492
3,2F5393B75558017AE0530A839427C6B4,Auffahrunfall,Unfall mit Leichtverletzten,false,false,false,Hauptstrasse,2682564,1251648,ZH,0261,2016,März,Freitag,17,17h-18h,T50,POINT (682564.000 251648.002),7.507529
4,303090D5AB8300A8E0530A83942731D6,Auffahrunfall,Unfall mit Leichtverletzten,false,false,false,Hauptstrasse,2685438,1251781,ZH,0261,2016,Januar,Donnerstag,10,10h-11h,T50,POINT (685438.000 251781.001),9.588702


In [9]:
# Maximum und Minimum der Distanzen
print(unfaelle['distance_to_light'].max())
print(unfaelle['distance_to_light'].min())


1279.143491890631
0.09398892528605553


In [10]:
# Erstelle eine Spalte keine_beleuchtung, die 1 ist wenn die Distanz grösser als 10 Meter ist, ansonsten 0
unfaelle['keine_beleuchtung'] = unfaelle['distance_to_light'] > 20

In [11]:
unfaelle['keine_beleuchtung'].value_counts()

keine_beleuchtung
False    11140
True       549
Name: count, dtype: int64

In [12]:
unfaelle = unfaelle.to_crs(epsg=4326)
beleuchtung = beleuchtung.to_crs(epsg=4326)

In [13]:
# Plotte die Unfälle auf einer Karte und färbe sie nach der neuen Spalte keine_beleuchtung
fig = px.scatter_mapbox(unfaelle, lat=unfaelle.geometry.y, lon=unfaelle.geometry.x, color='keine_beleuchtung', zoom=10)
fig.update_layout(
    mapbox_style="open-street-map", 
    width=600,
    height=600,
    legend=dict(
        x=0.01,  # Positioniert die Legende innerhalb der Karte
        y=0.99,  # nahe dem oberen Rand
        bgcolor="rgba(255, 255, 255, 0.8)"  # Halbtransparenter Hintergrund für bessere Lesbarkeit
    )
)
fig.show()

In [14]:
unfaelle.head()

,UnfallID,Unfallart,Unfallschwere,Fussgänger_beteiligt,Fahrrad_beteiligt,Motorrad_beteiligt,Strassentyp,Unfallort_Lon,Unfallort_Lat,Kanton,Gemeinde,Jahr,Monat,Wochentag,Stunde,Stunde_text,Temporegime,geometry,distance_to_light,keine_beleuchtung
0,2AF2FBACB62B00DAE0530A8394273CC3,Fussgängerunfall,Unfall mit Leichtverletzten,true,true,false,Nebenstrasse,2681186,1247896,ZH,0261,2016,Januar,Mittwoch,16,16h-17h,T50,POINT (8.51365 47.37690),1.563390,False
1,2B51E1C5A119015CE0530A839427323A,Schleuder- oder Selbstunfall,Unfall mit Leichtverletzten,false,true,false,Hauptstrasse,2682739,1244360,ZH,0261,2016,Januar,Freitag,13,13h-14h,T50,POINT (8.53356 47.34490),4.428940,False
2,2C6EC4B16319006EE0530A8394275C3E,Schleuder- oder Selbstunfall,Unfall mit Leichtverletzten,false,true,false,Nebenstrasse,2683598,1248099,ZH,0261,2016,Februar,Samstag,19,19h-20h,T50,POINT (8.54563 47.37842),5.570492,False
3,2F5393B75558017AE0530A839427C6B4,Auffahrunfall,Unfall mit Leichtverletzten,false,false,false,Hauptstrasse,2682564,1251648,ZH,0261,2016,März,Freitag,17,17h-18h,T50,POINT (8.53259 47.41047),7.507529,False
4,303090D5AB8300A8E0530A83942731D6,Auffahrunfall,Unfall mit Leichtverletzten,false,false,false,Hauptstrasse,2685438,1251781,ZH,0261,2016,Januar,Donnerstag,10,10h-11h,T50,POINT (8.57069 47.41130),9.588702,False


In [15]:
# Ändere die Werte der Spalte Temporegime
unfaelle['Temporegime'] = unfaelle['Temporegime'].replace({
    'T30': '30 km/h',
    'T20': '20 km/h',
    'T50': '50 km/h',
    'T60': '60 km/h',
    'T80': '80 km/h',
    'T100': '100 km/h',
    'T120': '120 km/h',
    'T30 tagsüber, Nachtfahrverbot': '30 km/h',
    'T50 tagsüber, T30 nachts': '50 km/h',
})

In [16]:
# Speichere die Daten
unfaelle.to_file(r"C:\Users\leoni\Documents\HSLU\5. Semester\Informationsvisualisierung\Einzelprojekt\informationsvisualisierung-leonid-wouters\Daten\unfaelle_zuerich_2023_temporegime_beleuchtung.json", driver='GeoJSON')